<a href="https://colab.research.google.com/github/Saishiva-hub/Rag-Agent-chatbot/blob/main/newsninja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install  -qU langchain-groq langgraph langchain-community tavily-python

In [ ]:
import os
import getpass

print('Enter your groq api key:')
os.environ['GROQ_API_KEY'] = getpass.getpass()

print('Enter your Tavily API key:')
os.environ['Tavily_API_KEY'] = getpass.getpass()

Enter your groq api key:
··········
Enter your Tavily API key:
··········


In [ ]:
import warnings
warnings.filterwarnings('ignore',category=DeprecationWarning)


from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

@tool
def search_new(query: str) -> str:
  '''Search the web for the latest news articles on a given topic, simply.'''
  search = TavilySearchResults(
      max_results = 5,
      search_depth = 'advanced',
      include_answer = True
  )
  results = search.invoke({'query':query})

  if not results:
    return 'No RESULTS FOUND FOR THE GIVEN QUERY'

  formatted = []
  for i , result in enumerate(results,1):
    title = result.get('title','No Title')
    url = result.get('url','')
    content = result.get('content','No content')
    formatted.append(
        f'**Ariticle {i}:** {title}\n'
        f'URL: {url}\n'
        f'Summary: {content}\n'
    )

  return '\n'.join(formatted)

In [ ]:
from typing import TypedDict
from datetime import datetime
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph , END



class AgentState(TypedDict):
  topic: str
  search_query : str
  search_results : str
  digest: str





def parse_topic(state: AgentState)-> dict:
  '''parse & refine user topic into an optimized search query.'''
  llm = ChatGroq(model = 'openai/gpt-oss-120b',
                 temperature = 0)
  prompt = (
      " You are a search query optimizer. Give the following topic, create a conise, "
      " effective search query to find the latest new about it."
      " Return only the optimized search query, nothing else.\n\n"
      f"Topic': {state['topic']}"
  )
  response = llm.invoke(prompt)
  search_query = response.content.strip()
  print(f"\n[Node: parse_topic] optimised seach query: {search_query}")
  return {'search_query': search_query}



def search_web(state: AgentState)-> dict:
  """Use Tavily to search for the news article"""
  query = state.get('search_query',state['topic'])
  print(f'\n[Node: search_web] searching the web for: {query}')
  results = search_new.invoke({'query': query})
  print(f"Found {results.count('Article')} articles")
  return {'search_results': results}


def write_digest(state: AgentState)-> dict:
  """ Use groq LLM to syntesize a polished news digest."""
  llm = ChatGroq(model = 'openi/gpt-oss-120b',
                 temperature=0)
  prompt = (
      "you are NewsNinja, an elite AI news analydt. Based on the following search "
      "resluts. create a comprehensive yet concise news digest. \n\n"
      "FORMAT YOUR RESPONSE AS: \n"
      "** NewsNinja Digest **\n\n"
      f"**Topic: ** {state['topic']}\n"
      f"**Dtae: ** {datetime.now().strftime('%B %d, %Y')}\n\n"
      "---------------------------------------\n\n"
      "Then provide: \n\n"
      "1. A brief **Executive Summary** (2-3 sentences)\n"
      "2. **key Headlines** with bullet points\n"
      "3. **Analysis & Insights** (what this means)\n"
      "4. **Sources** (list the URLs)\n\n"
      "---------------------------------------\n\n"
      f"SEARCH RESULTS: \n{state['search_results']}"
  )
  response = llm.invoke(prompt)
  print('\n[Node: write_digest] Digest written!!!')
  return {'digest': response.content}


def build_graph() -> StateGraph:
  graph = StateGraph(AgentState)
  graph.add_node('parser',parse_topic)
  graph.add_node('searcher',search_web)
  graph.add_node('writer',write_digest)

  graph.set_entry_point('parser')
  graph.add_edge('parser','searcher')
  graph.add_edge('searcher','writer')
  graph.add_edge('writer',END)

  return graph.compile()


app = build_graph()

In [ ]:
topic = input('Enter a new s topic to research: ')

if topic.strip():
  print('='*60)
  print(f'Starting pipeline for: {topic}')
  print('-'*60)


  result = app.invoke({'topic': topic})

  print('\n' + '='*60)
  print(result['digest'])
  print('='*60)

else:
  print('NO topic is provided. please try again')

Enter a new s topic to research: cricket
Starting pipeline for: cricket
------------------------------------------------------------

[Node: parse_topic] optimised seach query: cricket latest news 2024

[Node: search_web] searching the web for: cricket latest news 2024
Found 0 articles


NotFoundError: Error code: 404 - {'error': {'message': 'The model `openi/gpt-oss-120b` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}